In [ ]:
from google.colab import drive
drive.mount('data')

Mounted at data


In [ ]:
pip install pandas numpy opencv-python matplotlib

In [ ]:
import pandas as pd
import os

extracted_folder = '/content/dataset'
csv_path = None

# 1. Automatically search for the driving_log.csv file
for root, dirs, files in os.walk(extracted_folder):
    if 'driving_log.csv' in files:
        csv_path = os.path.join(root, 'driving_log.csv')
        # Also capture the exact directory where the CSV was found
        # (The IMG folder is usually in the same directory as the CSV)
        base_dir = root
        break

if csv_path is None:
    raise FileNotFoundError("Could not find 'driving_log.csv' inside the extracted folder. Please check your zip file contents.")

print(f"Successfully found CSV at: {csv_path}")

# 2. Load the CSV
columns = ['center', 'left', 'right', 'steering', 'throttle', 'reverse', 'speed']
df = pd.read_csv(csv_path, names=columns)

# 3. Drop left and right camera columns
df = df[['center', 'steering']]

# 4. Fix file paths to point to your unzipped Colab directory
# The simulator saves paths like 'C:\...\IMG\center_2024.jpg'.
# We just extract the filename and append it to our local Colab IMG path.
img_folder_path = os.path.join(base_dir, 'IMG')
df['center'] = df['center'].apply(lambda x: os.path.join(img_folder_path, os.path.basename(x.strip().replace('\\', '/'))))

print("Dataset shape after dropping left/right cameras:", df.shape)
print(df.head())

Successfully found CSV at: /content/dataset/data/driving_log.csv
Dataset shape after dropping left/right cameras: (6146, 2)
                                              center  steering
0  /content/dataset/data/IMG/center_2026_08_04_10...       0.0
1  /content/dataset/data/IMG/center_2026_08_04_10...       0.0
2  /content/dataset/data/IMG/center_2026_08_04_10...       0.0
3  /content/dataset/data/IMG/center_2026_08_04_10...       0.0
4  /content/dataset/data/IMG/center_2026_08_04_10...       0.0


In [ ]:
import cv2
import numpy as np

def preprocess_image(image_path):
    # Read image using OpenCV
    img = cv2.imread(image_path)

    # Convert BGR (OpenCV default) to RGB
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Crop: remove top 60px (sky/scenery) and bottom 25px (car hood)
    img = img[60:135, :, :]

    # Resize to Nvidia CNN architecture standard (200x66)
    img = cv2.resize(img, (200, 66))

    # Normalize pixel values to [0.0, 1.0]
    img = img / 255.0
    return img

def augment_image(image_path, steering_angle):
    img = preprocess_image(image_path)

    # 50% chance to flip image horizontally to compensate for forward-only lap bias
    if np.random.rand() > 0.5:
        img = cv2.flip(img, 1)
        steering_angle = -steering_angle

    return img, steering_angle

# Test preprocessing on the first center image
test_img, test_steering = augment_image(df['center'].iloc[0], df['steering'].iloc[0])
print("Processed center image shape:", test_img.shape)

Processed center image shape: (66, 200, 3)


In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np

# Split data into 80% training and 20% validation sets
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

def batch_generator(dataframe, batch_size=32, is_training=True):
    num_samples = len(dataframe)
    while True: # Loop forever so the generator never terminates
        dataframe = dataframe.sample(frac=1).reset_index(drop=True) # Shuffle each epoch
        for offset in range(0, num_samples, batch_size):
            batch_samples = dataframe.iloc[offset:offset+batch_size]

            images = []
            angles = []

            for _, row in batch_samples.iterrows():
                path = row['center']
                angle = float(row['steering'])

                if is_training:
                    # Apply dynamic augmentation (flip) for training data
                    img, angle = augment_image(path, angle)
                else:
                    # Pure preprocessing for validation data
                    img = preprocess_image(path)

                images.append(img)
                angles.append(angle)

            X_batch = np.array(images)
            y_batch = np.array(angles)

            yield X_batch, y_batch

# Test generator
train_gen = batch_generator(train_df, batch_size=32, is_training=True)
val_gen = batch_generator(val_df, batch_size=32, is_training=False)

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, Flatten, Dense, Dropout

def build_nvidia_model():
    model = Sequential([
        # Convolutional Layers
        Conv2D(24, (5, 5), strides=(2, 2), activation='relu', input_shape=(66, 200, 3)),
        Conv2D(36, (5, 5), strides=(2, 2), activation='relu'),
        Conv2D(48, (5, 5), strides=(2, 2), activation='relu'),
        Conv2D(64, (3, 3), activation='relu'),
        Conv2D(64, (3, 3), activation='relu'),

        Dropout(0.5), # Helps prevent overfitting

        # Fully Connected Layers
        Flatten(),
        Dense(100, activation='relu'),
        Dense(50, activation='relu'),
        Dense(10, activation='relu'),
        Dense(1) # Predicts single continuous value: steering angle
    ])

    model.compile(optimizer='adam', loss='mse')
    return model

model = build_nvidia_model()
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 31, 98, 24)     │         1,824 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 14, 47, 36)     │        21,636 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 5, 22, 48)      │        43,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 3, 20, 64)      │        27,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 1, 18, 64)      │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1, 18, 64)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1152)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 100)            │       115,300 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 50)             │         5,050 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │           510 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            11 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 252,219 (985.23 KB)

 Trainable params: 252,219 (985.23 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
batch_size = 32
epochs = 10

steps_per_epoch = len(train_df) // batch_size
validation_steps = len(val_df) // batch_size

history = model.fit(
    train_gen,
    steps_per_epoch=steps_per_epoch,
    epochs=epochs,
    validation_data=val_gen,
    validation_steps=validation_steps,
    verbose=1
)

# Save the trained model file
model.save('model.h5')
print("Training complete! Saved trained model as model.h5")

Epoch 1/10
153/153 ━━━━━━━━━━━━━━━━━━━━ 48s 285ms/step - loss: 0.0203 - val_loss: 0.0214
Epoch 2/10
153/153 ━━━━━━━━━━━━━━━━━━━━ 82s 538ms/step - loss: 0.0201 - val_loss: 0.0197
Epoch 3/10
153/153 ━━━━━━━━━━━━━━━━━━━━ 82s 535ms/step - loss: 0.0182 - val_loss: 0.0169
Epoch 4/10
153/153 ━━━━━━━━━━━━━━━━━━━━ 83s 544ms/step - loss: 0.0170 - val_loss: 0.0161
Epoch 5/10
153/153 ━━━━━━━━━━━━━━━━━━━━ 81s 531ms/step - loss: 0.0159 - val_loss: 0.0158
Epoch 6/10
153/153 ━━━━━━━━━━━━━━━━━━━━ 45s 293ms/step - loss: 0.0153 - val_loss: 0.0150
Epoch 7/10
153/153 ━━━━━━━━━━━━━━━━━━━━ 84s 552ms/step - loss: 0.0148 - val_loss: 0.0157
Epoch 8/10
153/153 ━━━━━━━━━━━━━━━━━━━━ 80s 522ms/step - loss: 0.0141 - val_loss: 0.0146
Epoch 9/10
153/153 ━━━━━━━━━━━━━━━━━━━━ 85s 555ms/step - loss: 0.0144 - val_loss: 0.0126
Epoch 10/10
153/153 ━━━━━━━━━━━━━━━━━━━━ 79s 520ms/step - loss: 0.0143 - val_loss: 0.0196


Training complete! Saved trained model as model.h5


In [ ]:
from google.colab import files
files.download('model.h5')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>